# TP4. Matrices et systèmes linéaires

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp4-matrices-gauss/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : la transposée

Écrivez `transposee(A)` qui échange lignes et colonnes.

**Résultat attendu :** la transposée de `[[1, 2, 3], [4, 5, 6]]` est
`[[1, 4], [2, 5], [3, 6]]`.


In [1]:
def transposee(A):
    lignes = len(A)
    colonnes = len(A[0])
    T = []
    for j in range(colonnes):
        ligne = []
        for i in range(lignes):
            ligne.append(A[i][j])
        T.append(ligne)
    return T

print(transposee([[1, 2, 3], [4, 5, 6]]))

[[1, 4], [2, 5], [3, 6]]


## Exercice 2.1 : produit à la main

Écrivez `produit(A, B)`, et vérifiez sur un exemple que vous savez calculer de tête.

**Résultat attendu :** multiplier une matrice par l'identité la laisse inchangée.


In [2]:
def produit(A, B):
    n = len(A)
    m = len(B[0])
    p = len(B)
    C = []
    for i in range(n):
        ligne = []
        for j in range(m):
            total = 0
            for k in range(p):
                total = total + A[i][k] * B[k][j]
            ligne.append(total)
        C.append(ligne)
    return C

A = [[1, 2], [3, 4]]
I = [[1, 0], [0, 1]]
print(produit(A, I))
print(produit(I, A))
print(produit(A, [[2, 0], [0, 2]]))

[[1, 2], [3, 4]]
[[1, 2], [3, 4]]
[[2, 4], [6, 8]]


## Exercice 2.2 : mesurer le coût cubique

Mesurez le temps de `produit` sur deux matrices carrées aléatoires de taille 40, 80
puis 120. Que devient le temps quand la taille double ?

**Résultat attendu :** le temps est multiplié par environ 8 quand la taille double.


In [3]:
import random
import time

for n in [40, 80, 120]:
    A = []
    B = []
    for i in range(n):
        A.append([random.random() for j in range(n)])
        B.append([random.random() for j in range(n)])
    debut = time.perf_counter()
    produit(A, B)
    print(n, round(time.perf_counter() - debut, 4))

40 0.0025
80 0.0218
120 0.072


## Exercice 3.1 : le rapport de temps

Reprenez la mesure de l'exercice 2.2 pour `n = 120`, et comparez au produit numpy des
mêmes matrices.

**Résultat attendu :** numpy est plus rapide de deux à trois ordres de grandeur.


In [4]:
import numpy as np

n = 120
A = []
B = []
for i in range(n):
    A.append([random.random() for j in range(n)])
    B.append([random.random() for j in range(n)])

debut = time.perf_counter()
produit(A, B)
t_python = time.perf_counter() - debut

NA = np.array(A)
NB = np.array(B)
debut = time.perf_counter()
for essai in range(100):
    NA @ NB
t_numpy = (time.perf_counter() - debut) / 100

print(t_python, t_numpy, t_python / t_numpy)

0.07977988100000744 9.934097000012798e-05 803.0914234067238


## Exercice 4.1 : élimination et remontée

Écrivez `resoudre(A, b)` qui résout le système par le pivot de Gauss avec pivot
partiel, et testez-la sur le système de l'étape 1.

**Résultat attendu :** la solution est $(3,\, -1,\, 0{,}5)$, et $A x$
redonne bien $b$.


In [5]:
def resoudre(A, b):
    n = len(A)
    M = []
    for ligne in A:
        M.append(ligne[:])
    y = b[:]
    for k in range(n):
        p = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[p][k]):
                p = i
        M[k], M[p] = M[p], M[k]
        y[k], y[p] = y[p], y[k]
        if M[k][k] == 0:
            raise ValueError("système singulier")
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] = M[i][j] - facteur * M[k][j]
            y[i] = y[i] - facteur * y[k]
    x = [0] * n
    for i in range(n - 1, -1, -1):
        s = y[i]
        for j in range(i + 1, n):
            s = s - M[i][j] * x[j]
        x[i] = s / M[i][i]
    return x

A = [[0, 1, 2], [1, 2, 2], [1, 1, 2]]
b = [0, 2, 3]
x = resoudre(A, b)
print(x)
print(produit(A, [[v] for v in x]))

[3.0, -1.0, 0.5]
[[0.0], [2.0], [3.0]]


## Exercice 4.2 : confronter à numpy

Comparez votre solution à celle de `numpy.linalg.solve` sur un système aléatoire de
taille 50.

**Résultat attendu :** les deux solutions coïncident à $10^{-9}$ près.


In [6]:
A = np.random.rand(50, 50)
b = np.random.rand(50)
x1 = resoudre(A.tolist(), b.tolist())
x2 = np.linalg.solve(A, b)
print(np.max(np.abs(np.array(x1) - x2)))

8.715250743307479e-15


## Exercice 5.1 : lire le résultat

Exécutez le bloc ci-dessus et commentez. Quelle méthode donne la solution la plus
proche de la vraie ? Que vaut le conditionnement, et qu'est-ce que cela signifie ?


In [7]:
n = 12
H = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        H[i][j] = 1 / (i + j + 1)
b = H.sum(axis=1)

x_solve = np.linalg.solve(H, b)
x_inv = np.linalg.inv(H) @ b
print(np.max(np.abs(x_solve - 1)))
print(np.max(np.abs(x_inv - 1)))
print(np.linalg.cond(H))
print("Les deux méthodes se trompent. Le conditionnement est énorme, donc une toute petite erreur sur les données est multipliée par ce nombre dans la solution.")

0.5162080651308802
5.734375
1.6197474255692448e+16
Les deux méthodes se trompent. Le conditionnement est énorme, donc une toute petite erreur sur les données est multipliée par ce nombre dans la solution.


## Exercice 6.1 : composer deux transformations

Vérifiez numériquement que faire tourner de 30 degrés puis de 60 degrés revient à
faire tourner de 90 degrés, autrement dit que
$R(60°) \cdot R(30°) = R(90°)$.


In [8]:
import math

def rotation(angle):
    a = math.radians(angle)
    return np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])

print(rotation(60) @ rotation(30))
print(rotation(90))
print(np.allclose(rotation(60) @ rotation(30), rotation(90)))

[[ 2.14607521e-16 -1.00000000e+00]
 [ 1.00000000e+00  2.07170437e-16]]
[[ 6.123234e-17 -1.000000e+00]
 [ 1.000000e+00  6.123234e-17]]
True


## Exercice 7.1 : déterminant par Gauss

Écrivez `determinant(A)` en reprenant l'élimination de `resoudre`, sans second membre :
multipliez les pivots successifs, et changez le signe à chaque échange de lignes.
Comparez à `np.linalg.det`.

**Résultat attendu :** le déterminant de la matrice de l'étape 1 vaut `-2`, et celui
de `[[1, 2], [2, 4]]` vaut `0`.


In [9]:
def determinant(A):
    n = len(A)
    M = []
    for ligne in A:
        M.append(ligne[:])
    det = 1
    for k in range(n):
        p = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[p][k]):
                p = i
        if p != k:
            M[k], M[p] = M[p], M[k]
            det = -det
        if M[k][k] == 0:
            return 0
        det = det * M[k][k]
        for i in range(k + 1, n):
            facteur = M[i][k] / M[k][k]
            for j in range(k, n):
                M[i][j] = M[i][j] - facteur * M[k][j]
    return det

A = [[0, 1, 2], [1, 2, 2], [1, 1, 2]]
print(determinant(A), np.linalg.det(np.array(A)))
print(determinant([[1, 2], [2, 4]]), np.linalg.det(np.array([[1, 2], [2, 4]])))

-2.0 -2.0
0 0.0


## Exercice 8.1 : Hilbert vaincue

Construisez la matrice de Hilbert de taille 12 avec des `Fraction(1, i + j + 1)`, le
second membre `b` comme somme des lignes, pour que la solution soit le vecteur de 1,
puis appelez **votre** fonction `resoudre` de l'étape 4 dessus. Le premier essai échoue
avec « système singulier ». Comprenez pourquoi avant de lire le corrigé, puis corrigez.

**Résultat attendu :** exactement `[1, 1, ..., 1]`, douze fois `Fraction(1, 1)`, là
où `solve` se trompait de 0,7.


In [10]:
from fractions import Fraction

n = 12
H = []
for i in range(n):
    ligne = []
    for j in range(n):
        ligne.append(Fraction(1, i + j + 1))
    H.append(ligne)

b = []
for i in range(n):
    b.append(sum(H[i]))

print(resoudre(H, b))

[Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1), Fraction(1, 1)]


## Exercice 9.1 : l'ordre compte

Écrivez `translation(tx, ty)` et `rotation_h(angle)` qui renvoient des matrices
$3 \times 3$. Appliquez au point $(1, 0)$ la translation de $(3, 1)$ suivie
de la rotation de 90°, puis la rotation suivie de la translation.

**Résultat attendu :** $(-1, 4)$ dans le premier cas, $(3, 2)$ dans le second.
Les deux résultats diffèrent.


In [11]:
def translation(tx, ty):
    return np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]])

def rotation_h(angle):
    a = math.radians(angle)
    return np.array([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]])

point = np.array([1, 0, 1])
T = translation(3, 1)
R = rotation_h(90)
print(np.round(R @ (T @ point), 6))
print(np.round(T @ (R @ point), 6))

[-1.  4.  1.]
[3. 2. 1.]
